# 00 - Audit of inputs and sources (clinical_outcome_v2)

**Gate 0.** Nothing downstream may run until this notebook completes with `gate0_passed = True`.

Scope:
1. Prove every output path is new and that no legacy result directory is written into.
2. Inventory + SHA-256 every input (H5AD, CellPhoneDB database, config, `clinical_info.csv`, `legacy_sources/`).
3. Record the exact software / hardware environment, including the missing Parquet/Excel prerequisites.
4. Validate cell metadata, the patient -> outcome mapping, and the raw count matrix **without densifying `X`**.
5. Establish gene-identifier semantics for `counts_data='hgnc_symbol'`.
6. Emit the 24 x 19 per-sample cell-type abundance matrix and the core cell-type set.
7. Audit the available clinical covariates and record what is *not* available.

Rules obeyed here: assert rather than print; never hardcode a patient list; never read `obs['batch_1']`
after the single forensic cell that documents why it is wrong; never load `adata.X` (dense float64,
~7.6 GiB, z-scored and unusable).

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
# ============================== CONFIG CELL (the only one) ==============================
import os, sys, json, platform, shutil, zipfile, io, re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import h5py

PROJECT_DIR = Path(os.environ.get(
    "CPDB_V2_DIR",
    f"{ACC_DATA_ROOT}/scripts/named_scripts/cellphoneDB_scripts/clinical_outcome_v2",
)).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

CONFIG_PATH = PROJECT_DIR / "config.yaml"

from utils import (load_config, config_hash, sha256_file, write_manifest, log_jsonl,
                   derive_patient_outcomes, celltype_abundance_matrix, pooled_retained_celltypes,
                   balance_downsample, patient_retained_celltypes, core_celltypes,
                   validate_counts_matrix, write_cpdb_input)

CFG = load_config(CONFIG_PATH)
CFG_HASH = config_hash(CFG)

H5AD_PATH       = Path(CFG["input_h5ad"])
CPDB_DB_PATH    = Path(CFG["cpdb_database"])
PROJECT_ROOT    = Path(CFG["project_root"])
OUTPUT_ROOT     = Path(CFG["output_root"])
SAMPLE_COL      = CFG["sample_column"]
CELLTYPE_COL    = CFG["cell_type_column"]
OUTCOME_COL     = CFG["outcome_column"]
COUNTS_LAYER    = CFG["counts_layer"]
OUTCOMES        = list(CFG.get("outcomes", ["Good", "Poor"]))
PATIENT_MIN_CELLS  = int(CFG["patient_celltype_min_cells"])
POOLED_MIN_PER_GRP = int(CFG["pooled_celltype_min_per_group"])
POOLED_SEEDS       = [int(s) for s in CFG["pooled_downsample_seeds"]]
CORE_MIN_PATIENTS  = int(CFG["secondary_core_min_patients"])

CLINICAL_CSV = Path(CFG.get("covariate_csv", PROJECT_ROOT / "clinical_info.csv"))
SITE_COL     = CFG.get("covariate_column_map", {}).get("site_of_extraction", "Site of Extraction")
FORBIDDEN_OBS_COLS = list(CFG.get("forbidden_obs_columns", ["batch_1", "site", "model_outcome", "outcome"]))
LEGACY_DIR   = PROJECT_DIR / "legacy_sources"

AUDIT_DIR     = OUTPUT_ROOT / "audit"
LOG_DIR       = OUTPUT_ROOT / "logs"
MANIFEST_PATH = OUTPUT_ROOT / "MANIFEST.sha256"
EVENT_LOG     = LOG_DIR / "00_audit_events.jsonl"

# Locations this project must never write into.
FORBIDDEN_SUBSTRINGS = ("cpdb_results_", "t1_cpdb_results", "t2_cpdb_results", "t3_cpdb_results",
                        "cellphoneDB_copy_backup")

# ---- Expected facts. These are ASSERTED, never imposed on the data. --------------------
EXPECTED = dict(
    n_cells=95441,
    n_patients=24,
    n_celltypes=19,
    n_good_patients=13,
    n_poor_patients=11,
    n_good_cells=46260,
    n_poor_cells=49181,
    outcome_levels={"Good", "Poor"},
    core_celltypes=["ADC - Tumor", "ADMEC - Tumor", "Endothelial cells",
                    "Epithelial cells - Tumor", "Fibroblast cells", "Macrophages - M2"],
    core_next_ranked={"Mural cells": 19, "CD4+ T cells": 19},
    core_patient_support={"Fibroblast cells": 24, "Epithelial cells - Tumor": 24,
                          "ADC - Tumor": 23, "Macrophages - M2": 23,
                          "ADMEC - Tumor": 23, "Endothelial cells": 22},
    clinical_columns=["Sample ID", "Site of Extraction", "Years", "Clinical outcome",
                      "Model outcome", "Bulk RNA Datasets", "Single Cell RNA Datasets",
                      "Spatial Datasets"],
    site_by_outcome={"Maxilla": (0, 4), "Parotid Gland": (3, 0), "Larynx": (2, 0),
                     "Tongue": (2, 0), "Unknown": (4, 1), "Base of Tongue": (2, 2)},
    years_levels={2, 3, 5},
    years_to_outcome={2: "Poor", 3: "Good", 5: "Good"},
    clinical_only_patient="P14",
    batch1_mislabelled_patient="P07",
    model_outcome_disagreements=["P15", "P17", "P20", "P24"],
    obs_site_labels=11,
    csv_site_labels=10,
    obs_site_split_pairs={"BOT": "Good", "BaseofTongue": "Poor"},
)

RUN_STARTED = datetime.now(timezone.utc).isoformat()
CHUNK_ROWS = 10_000     # rows per chunk when validating the sparse counts layer

In [ ]:
# ---- Recording helpers (no analysis logic lives here; utils.py owns that) -------------
CHECKS = []    # every assertion, blocking or not
TABLES = {}    # audit tables written to CSV at the end
NOTES  = []    # narrative limitations carried into the audit report


def log_event(record):
    log_jsonl(EVENT_LOG, {"ts": datetime.now(timezone.utc).isoformat(),
                          "notebook": "00_audit", **record})


def check(cid, description, ok, detail=None, blocking=True):
    """Record an assertion result; raise immediately when a blocking check fails."""
    ok = bool(ok)
    entry = {"id": cid, "description": description, "passed": ok,
             "blocking": blocking, "detail": detail}
    CHECKS.append(entry)
    log_event({"event": "check", **entry})
    if not ok and blocking:
        raise AssertionError(f"[{cid}] {description} FAILED :: {detail}")
    return ok


def note(text):
    NOTES.append(text)
    log_event({"event": "note", "text": text})


def assert_safe_path(path):
    p = str(Path(path).resolve())
    bad = [s for s in FORBIDDEN_SUBSTRINGS if s in p]
    if bad:
        raise AssertionError(f"Refusing to touch protected location {p} (matched {bad})")
    return Path(path)


def as_str_list(obj):
    if isinstance(obj, (pd.Series, pd.Index)):
        obj = obj.tolist()
    return sorted(str(x) for x in obj)


def human_bytes(n):
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024:
            return f"{n:,.1f} {unit}"
        n /= 1024
    return f"{n:,.1f} PiB"


LOG_DIR.mkdir(parents=True, exist_ok=True)
log_event({"event": "start", "config_path": str(CONFIG_PATH), "config_hash": CFG_HASH,
           "started": RUN_STARTED})

## 1. Output locations are new, and no legacy result directory is in scope

Every path this project writes to is created here. A path is acceptable only if it holds no files
yet, and is rejected outright if it lives inside `cpdb_results_*`, `t1/t2/t3_*` or
`cellphoneDB_copy_backup/`.

In [ ]:
OUTPUT_SUBDIRS = ["audit", "inputs", "logs", "cpdb_runs", "primary", "sensitivity",
                  "exploratory_pooled", "figures"]

check("A00.01", "output_root is the clinical_outcome_v2 tree",
      OUTPUT_ROOT.name == "clinical_outcome_v2", detail=str(OUTPUT_ROOT))

assert_safe_path(OUTPUT_ROOT)
for sub in OUTPUT_SUBDIRS:
    assert_safe_path(OUTPUT_ROOT / sub)

preexisting = {}
for sub in OUTPUT_SUBDIRS:
    d = OUTPUT_ROOT / sub
    preexisting[sub] = sorted(str(p.relative_to(OUTPUT_ROOT))
                              for p in d.rglob("*") if p.is_file()) if d.exists() else []

# The audit/logs directories may already contain this notebook's own event log if it is re-run.
external = {k: [f for f in v if not f.startswith(("audit/", "logs/"))] for k, v in preexisting.items()}

check("A00.02", "no pre-existing analysis files inside any output subdirectory",
      all(len(v) == 0 for v in external.values()),
      detail={k: v[:10] for k, v in external.items() if v})
check("A00.03", "no output path resolves inside a protected legacy directory",
      all(not any(s in str((OUTPUT_ROOT / sub).resolve()) for s in FORBIDDEN_SUBSTRINGS)
          for sub in OUTPUT_SUBDIRS), detail=str(OUTPUT_ROOT))

for sub in OUTPUT_SUBDIRS:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

check("A00.04", "inputs/ is empty (notebook 01 will populate it)",
      not any((OUTPUT_ROOT / "inputs").rglob("*")), detail=preexisting["inputs"][:10])

TABLES["output_paths"] = pd.DataFrame(
    {"subdir": OUTPUT_SUBDIRS,
     "path": [str(OUTPUT_ROOT / s) for s in OUTPUT_SUBDIRS],
     "preexisting_files": [len(preexisting[s]) for s in OUTPUT_SUBDIRS]})
TABLES["output_paths"]

## 2. Input inventory and SHA-256

Inputs are the H5AD, the CellPhoneDB v5 database zip, `config.yaml`, `config_schema.json`,
`utils.py`, `clinical_info.csv`, and the five legacy scripts copied into `legacy_sources/`. Legacy
checksums are cross-checked against `legacy_sources/SOURCES.sha256`.

In [ ]:
INPUT_FILES = {
    "input_h5ad": H5AD_PATH,
    "cpdb_database": CPDB_DB_PATH,
    "config_yaml": CONFIG_PATH,
    "config_schema": PROJECT_DIR / "config_schema.json",
    "utils_py": PROJECT_DIR / "utils.py",
    "clinical_info_csv": CLINICAL_CSV,
}
legacy_files = sorted(p for p in LEGACY_DIR.glob("*.py"))
for p in legacy_files:
    INPUT_FILES[f"legacy_sources/{p.name}"] = p

missing = {k: str(v) for k, v in INPUT_FILES.items() if not Path(v).is_file()}
check("A00.05", "every declared input file exists", not missing, detail=missing)
check("A00.06", "legacy_sources contains exactly 5 python scripts",
      len(legacy_files) == 5, detail=[p.name for p in legacy_files])

rows = []
for key, path in INPUT_FILES.items():
    path, st = Path(path), Path(path).stat()
    rows.append({"key": key, "path": str(path), "bytes": st.st_size,
                 "size_human": human_bytes(st.st_size),
                 "mtime_utc": datetime.fromtimestamp(st.st_mtime, timezone.utc).isoformat(),
                 "sha256": sha256_file(path)})
inventory = pd.DataFrame(rows).set_index("key")
TABLES["input_inventory"] = inventory.reset_index()

check("A00.07", "no input file is empty", bool((inventory["bytes"] > 0).all()),
      detail=inventory["bytes"].to_dict())

sources_manifest = LEGACY_DIR / "SOURCES.sha256"
legacy_expected = {}
if sources_manifest.is_file():
    for line in sources_manifest.read_text().splitlines():
        line = line.strip()
        if line:
            digest, _, rel = line.partition("  ")
            legacy_expected[Path(rel).name] = digest
legacy_observed = {Path(inventory.loc[k, "path"]).name: inventory.loc[k, "sha256"]
                   for k in inventory.index if k.startswith("legacy_sources/")}
legacy_mismatch = {n: (legacy_expected.get(n), d) for n, d in legacy_observed.items()
                   if legacy_expected.get(n) != d}
check("A00.08", "legacy_sources checksums match SOURCES.sha256",
      len(legacy_expected) == 5 and not legacy_mismatch,
      detail={"recorded": len(legacy_expected), "mismatch": legacy_mismatch})

with zipfile.ZipFile(CPDB_DB_PATH) as zf:
    CPDB_DB_MEMBERS = sorted(zf.namelist())
check("A00.09", "CellPhoneDB zip exposes gene_table.csv and interaction_table.csv",
      {"gene_table.csv", "interaction_table.csv"}.issubset(set(CPDB_DB_MEMBERS)),
      detail=CPDB_DB_MEMBERS)

inventory[["size_human", "sha256"]]

## 3. Environment snapshot

In [ ]:
from importlib import metadata as importlib_metadata

PKGS = ["anndata", "scanpy", "cellphonedb", "pandas", "numpy", "scipy", "h5py", "matplotlib",
        "seaborn", "statsmodels", "ktplotspy", "PyYAML", "jsonschema", "scikit-learn", "numba",
        "networkx", "pyarrow", "openpyxl", "ipykernel", "jupyter_client", "psutil", "tqdm"]
pkg_versions = {}
for name in PKGS:
    try:
        pkg_versions[name] = importlib_metadata.version(name)
    except Exception:
        pkg_versions[name] = "NOT INSTALLED"

try:
    import psutil
    ram_total, ram_avail = psutil.virtual_memory().total, psutil.virtual_memory().available
    cpu_logical, cpu_physical = psutil.cpu_count(True), psutil.cpu_count(False)
except Exception:
    ram_total = ram_avail = -1
    cpu_logical, cpu_physical = os.cpu_count(), None
disk = shutil.disk_usage(OUTPUT_ROOT)

ENVIRONMENT = {
    "timestamp_utc": RUN_STARTED,
    "python_version": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
    "kernel_name": "cpdb",
    "kernel_display_name": "Python (cpdb)",
    "conda_prefix": os.environ.get("CONDA_PREFIX", ""),
    "platform": platform.platform(),
    "processor": platform.processor(),
    "hostname": platform.node(),
    "cpu_count_logical": cpu_logical,
    "cpu_count_physical": cpu_physical,
    "threads_configured": CFG.get("threads"),
    "ram_total": human_bytes(ram_total),
    "ram_available": human_bytes(ram_avail),
    "disk_total": human_bytes(disk.total),
    "disk_free": human_bytes(disk.free),
    "config_hash": CFG_HASH,
    "packages": pkg_versions,
}

snapshot_path = AUDIT_DIR / "environment_snapshot.txt"
lines = ["CellPhoneDB v5 clinical-outcome re-analysis - environment snapshot", "=" * 70]
lines += [f"{k:24s}: {v}" for k, v in ENVIRONMENT.items() if k != "packages"]
lines += ["", "packages", "-" * 70]
lines += [f"{k:24s}: {v}" for k, v in sorted(pkg_versions.items())]
snapshot_path.write_text("\n".join(lines) + "\n")

check("A00.10", "kernel python is the cpdb conda environment",
      "envs/cpdb" in sys.executable, detail=sys.executable)
check("A00.11", "environment_snapshot.txt written", snapshot_path.is_file(), detail=str(snapshot_path))
check("A00.12", "configured thread count does not exceed the logical CPU count",
      cpu_logical is None or int(CFG.get("threads", 1)) <= int(cpu_logical),
      detail={"threads": CFG.get("threads"), "cpus": cpu_logical}, blocking=False)
check("A00.13", "at least 8 GiB RAM available",
      ram_avail < 0 or ram_avail >= 8 * 1024**3, detail=human_bytes(ram_avail), blocking=False)
check("A00.14", "at least 100 GiB free disk under output_root",
      disk.free >= 100 * 1024**3, detail=human_bytes(disk.free), blocking=False)

### 3b. Prerequisite check: `pyarrow` and `openpyxl`

Parquet and Excel deliverables in the downstream notebooks need these two packages. This cell
**reports** their availability and deliberately installs nothing. Non-blocking for Gate 0 (data
validity is unaffected); the reporting notebooks fall back to CSV wherever Parquet is unavailable.

Both were absent from the `cpdb` environment when this project was planned. If they are present at
run time the check simply records the version - the point is that Gate 0 states the fact rather than
assuming it.


In [ ]:
PREREQ_PKGS = ["pyarrow", "openpyxl"]
prereq_status = {}
for name in PREREQ_PKGS:
    try:
        __import__(name)
        prereq_status[name] = importlib_metadata.version(name)
    except Exception as exc:
        prereq_status[name] = f"MISSING ({type(exc).__name__})"
PREREQ_MISSING = [k for k, v in prereq_status.items() if str(v).startswith("MISSING")]

check("A00.15", "pyarrow importable (needed for Parquet deliverables)",
      "pyarrow" not in PREREQ_MISSING, detail=prereq_status["pyarrow"], blocking=False)
check("A00.16", "openpyxl importable (needed for Excel deliverables)",
      "openpyxl" not in PREREQ_MISSING, detail=prereq_status["openpyxl"], blocking=False)

if PREREQ_MISSING:
    print("\n" + "!" * 78)
    print("PREREQUISITE NOT SATISFIED - human action required (nothing was installed)")
    print("!" * 78)
    print(f"Missing from the `cpdb` environment : {', '.join(PREREQ_MISSING)}")
    print("Required by                         : .parquet and .xlsx deliverables in the reporting notebooks")
    print("Impact on this notebook             : none; all data validation below is unaffected")
    print("Suggested fix (run manually)        : "
          f"conda install -n cpdb -c conda-forge {' '.join(PREREQ_MISSING)}")
    print("!" * 78)
    note(f"PREREQUISITE: {', '.join(PREREQ_MISSING)} are missing from the cpdb environment; "
         "Parquet/Excel deliverables cannot be produced until they are installed. This notebook "
         "reports the gap and installs nothing.")

TABLES["prerequisites"] = pd.DataFrame({"package": list(prereq_status),
                                        "status": list(prereq_status.values()),
                                        "required_for": ["Parquet deliverables", "Excel deliverables"]})
TABLES["prerequisites"]

## 4. H5AD metadata (backed mode - `X` is never materialised)

`X` is a **dense float64 (95441 x 10000)** array of z-scored values (~7.6 GiB, essentially the whole
file). It is unusable for CellPhoneDB and is never read here or anywhere downstream. All expression
work uses `layers['counts']` (CSR float32).

In [ ]:
adata = ad.read_h5ad(H5AD_PATH, backed="r")
obs = adata.obs.copy()      # metadata only
var = adata.var.copy()
n_cells, n_genes = adata.shape

check("A00.17", "H5AD cell count matches expectation",
      n_cells == EXPECTED["n_cells"], detail={"observed": n_cells, "expected": EXPECTED["n_cells"]})

REQUIRED_OBS = [SAMPLE_COL, CELLTYPE_COL, OUTCOME_COL]
check("A00.18", "required obs columns present", all(c in obs.columns for c in REQUIRED_OBS),
      detail={"missing": [c for c in REQUIRED_OBS if c not in obs.columns]})
check("A00.19", f"layers['{COUNTS_LAYER}'] exists", COUNTS_LAYER in list(adata.layers.keys()),
      detail=list(adata.layers.keys()))

with h5py.File(H5AD_PATH, "r") as h5:
    x_encoding = dict(h5["X"].attrs).get("encoding-type", "unknown")
    x_dtype = str(h5["X"].dtype) if hasattr(h5["X"], "dtype") else "group"
    counts_nnz = int(h5[f"layers/{COUNTS_LAYER}/data"].shape[0])
    counts_dtype = str(h5[f"layers/{COUNTS_LAYER}/data"].dtype)
    counts_encoding = dict(h5[f"layers/{COUNTS_LAYER}"].attrs).get("encoding-type", "unknown")

check("A00.20", "counts layer is stored sparse (CSR)", "csr" in str(counts_encoding).lower(),
      detail=counts_encoding)
check("A00.21", "counts layer dtype is float32", counts_dtype == "float32", detail=counts_dtype,
      blocking=False)

note(f"adata.X is {x_encoding} {x_dtype} of shape {tuple(adata.shape)} (z-scored, ~7.6 GiB dense). "
     f"It is never read by this project; expression comes from layers['{COUNTS_LAYER}'] "
     f"({counts_encoding}, {counts_dtype}, nnz={counts_nnz:,}).")

MATRIX_INFO = {"shape": [int(n_cells), int(n_genes)], "X_encoding": x_encoding, "X_dtype": x_dtype,
               "counts_encoding": counts_encoding, "counts_dtype": counts_dtype,
               "counts_nnz": counts_nnz,
               "counts_density": round(counts_nnz / (n_cells * n_genes), 4)}
MATRIX_INFO

In [ ]:
# ---- Label completeness, uniqueness, category counts --------------------------------
for col in REQUIRED_OBS:
    n_missing = int(obs[col].isna().sum())
    n_blank = int(obs[col].astype(str).str.strip().isin(["", "nan", "None"]).sum())
    check(f"A00.22.{col}", f"obs['{col}'] has no missing or blank labels",
          n_missing == 0 and n_blank == 0, detail={"nan": n_missing, "blank": n_blank})

patients = pd.Index(as_str_list(obs[SAMPLE_COL].unique()))
celltypes = pd.Index(as_str_list(obs[CELLTYPE_COL].unique()))

check("A00.23", "24 unique patients", len(patients) == EXPECTED["n_patients"],
      detail={"observed": len(patients), "patients": list(patients)})
check("A00.24", f"19 unique {CELLTYPE_COL} cell types", len(celltypes) == EXPECTED["n_celltypes"],
      detail={"observed": len(celltypes), "celltypes": list(celltypes)})
check("A00.25", "cell barcodes (obs_names) are unique", obs.index.is_unique,
      detail={"n": len(obs.index), "n_unique": int(obs.index.nunique())})
check("A00.26", "gene identifiers (var_names) are unique", var.index.is_unique,
      detail={"n": len(var.index), "n_unique": int(var.index.nunique())})
check("A00.27", "no barcode is empty or whitespace",
      bool((obs.index.astype(str).str.strip() != "").all()), detail=None)

## 5. Patient -> outcome mapping (derived at runtime, never hardcoded)

`utils.derive_patient_outcomes` raises if any patient carries more than one outcome, so a corrupt
join cannot pass silently. `obs['outcome']` is verified to be a duplicate of `clinical_outcome`.

In [ ]:
patient_outcomes = derive_patient_outcomes(obs, sample_col=SAMPLE_COL, outcome_col=OUTCOME_COL)

n_outcomes_per_patient = obs.groupby(SAMPLE_COL, observed=True)[OUTCOME_COL].nunique()
check("A00.28", "every patient maps to exactly one outcome",
      bool((n_outcomes_per_patient == 1).all()),
      detail=n_outcomes_per_patient[n_outcomes_per_patient != 1].to_dict())
check("A00.29", "derived mapping covers all 24 patients and no others",
      as_str_list(patient_outcomes.index) == list(patients), detail=len(patient_outcomes))
check("A00.30", "outcome levels are exactly {Good, Poor}",
      set(patient_outcomes.unique()) == EXPECTED["outcome_levels"],
      detail=as_str_list(patient_outcomes.unique()))
check("A00.31", "config 'outcomes' matches the observed levels",
      sorted(OUTCOMES) == sorted(EXPECTED["outcome_levels"]), detail=OUTCOMES)

GOOD_PATIENTS = as_str_list(patient_outcomes.index[patient_outcomes == "Good"])
POOR_PATIENTS = as_str_list(patient_outcomes.index[patient_outcomes == "Poor"])
n_good_cells = int((obs[OUTCOME_COL].astype(str) == "Good").sum())
n_poor_cells = int((obs[OUTCOME_COL].astype(str) == "Poor").sum())

check("A00.32", "13 Good and 11 Poor patients",
      len(GOOD_PATIENTS) == EXPECTED["n_good_patients"]
      and len(POOR_PATIENTS) == EXPECTED["n_poor_patients"],
      detail={"good": len(GOOD_PATIENTS), "poor": len(POOR_PATIENTS)})
check("A00.33", "46,260 Good cells and 49,181 Poor cells",
      (n_good_cells, n_poor_cells) == (EXPECTED["n_good_cells"], EXPECTED["n_poor_cells"]),
      detail={"good_cells": n_good_cells, "poor_cells": n_poor_cells})
check("A00.34", "Good + Poor cells account for every cell",
      n_good_cells + n_poor_cells == n_cells, detail=n_good_cells + n_poor_cells)
TABLES["patient_outcomes"] = (patient_outcomes.rename(OUTCOME_COL).rename_axis(SAMPLE_COL)
                              .reset_index()
                              .assign(n_cells=lambda d: d[SAMPLE_COL].map(
                                  obs[SAMPLE_COL].astype(str).value_counts())))
TABLES["patient_outcomes"]

### 5b. Forensic record: the forbidden `obs` columns, read once and retired

`config.yaml` lists `forbidden_obs_columns`. This is the **only** cell in the project that reads
them; it documents exactly why each is unusable, after which nothing downstream touches them.

* **`batch_1`** carries an outcome prefix per patient and labels **P07** as `poor_P07`, although
  that patient is clinically **Good** (3-year survival) - producing a bogus **12 Good / 12 Poor**
  split. (`utils.load_config` additionally refuses any config that names `batch_1`.)
* **`outcome`** is a duplicate of `clinical_outcome`; it is verified as such and then ignored, so
  there is only one outcome source of truth.
* **`model_outcome`** is the superseded model-predicted label; it disagrees with the clinical label
  for 4 patients, which is the entire reason for this re-analysis.
* **`site`** splits *Base of Tongue* into `BOT` (both patients Good) and `BaseofTongue` (both Poor),
  turning the only outcome-balanced site into two perfectly confounded ones, and codes Unknown as the
  string `"NA"`. Anatomical site is taken from `covariate_csv` instead.

In [ ]:
forensics = {}
patient_view = (obs.astype(str).groupby(SAMPLE_COL, observed=True)
                .agg(lambda s: sorted(set(s))[0]))

check("A00.35", "config declares the forbidden obs columns",
      set(FORBIDDEN_OBS_COLS) >= {"batch_1", "site", "model_outcome", "outcome"},
      detail=FORBIDDEN_OBS_COLS)

# ---- batch_1 -------------------------------------------------------------------------
if "batch_1" in obs.columns:
    b1 = (obs[[SAMPLE_COL, "batch_1", OUTCOME_COL]].astype(str)
          .groupby(SAMPLE_COL, observed=True)
          .agg(batch1=("batch_1", lambda s: sorted(set(s))[0]),
               clinical=(OUTCOME_COL, lambda s: sorted(set(s))[0])))
    b1["batch1_implied"] = b1["batch1"].str.split("_").str[0].str.capitalize()
    disagree = b1[b1["batch1_implied"] != b1["clinical"]]
    b1_counts = b1["batch1_implied"].value_counts().to_dict()

    check("A00.36", "batch_1 disagrees with clinical_outcome for exactly one patient",
          len(disagree) == 1, detail=disagree.to_dict("index"))
    check("A00.37", "the mislabelled patient is P07 (clinically Good, coded poor_)",
          list(disagree.index) == [EXPECTED["batch1_mislabelled_patient"]], detail=list(disagree.index))
    check("A00.38", "batch_1 would produce a bogus 12 Good / 12 Poor patient split",
          b1_counts.get("Good") == 12 and b1_counts.get("Poor") == 12, detail=b1_counts)
    forensics["batch_1"] = {"implied_patient_split": b1_counts,
                            "disagreements": disagree.to_dict("index"),
                            "status": "RETIRED"}
    TABLES["batch1_discrepancy"] = b1.reset_index()
    note("obs['batch_1'] mislabels P07 (clinically Good, 3-year survival) as 'poor_P07', "
         "producing a spurious 12/12 patient split. Retired: every grouping derives from "
         "obs['clinical_outcome'] (13 Good / 11 Poor).")

# ---- outcome (duplicate) --------------------------------------------------------------
check("A00.39", "obs['outcome'] duplicates obs['clinical_outcome'] (and is then ignored)",
      "outcome" in obs.columns
      and bool((obs["outcome"].astype(str) == obs[OUTCOME_COL].astype(str)).all()),
      detail={"n_disagree": int((obs.get("outcome", obs[OUTCOME_COL]).astype(str)
                                 != obs[OUTCOME_COL].astype(str)).sum())})
forensics["outcome"] = {"duplicate_of": OUTCOME_COL, "status": "RETIRED"}

# ---- model_outcome (superseded) -------------------------------------------------------
if "model_outcome" in obs.columns:
    mo_disagree = as_str_list(patient_view.index[patient_view["model_outcome"]
                                                 != patient_view[OUTCOME_COL]])
    check("A00.40", "model_outcome disagrees with clinical_outcome for exactly 4 patients",
          mo_disagree == sorted(EXPECTED["model_outcome_disagreements"]),
          detail={"disagreeing": mo_disagree})
    forensics["model_outcome"] = {"disagreeing_patients": mo_disagree,
                                  "status": "SUPERSEDED - clinical_outcome is authoritative"}
    note(f"obs['model_outcome'] (the superseded label) disagrees with clinical_outcome for "
         f"{len(mo_disagree)} patients ({', '.join(mo_disagree)}): all four are model-Good but "
         "clinically Poor. This re-analysis is rebased on clinical_outcome.")

# ---- site (must not be used raw) -------------------------------------------------------
if "site" in obs.columns:
    site_patients = (obs.assign(site=obs["site"].astype(str))
                     .groupby("site", observed=True)
                     .agg(patients=(SAMPLE_COL, lambda s: as_str_list(set(s))),
                          outcomes=(OUTCOME_COL, lambda s: as_str_list(set(s)))))
    check("A00.41", "obs['site'] carries 11 raw labels", len(site_patients) == EXPECTED["obs_site_labels"],
          detail=as_str_list(site_patients.index))
    check("A00.42", "obs['site'] splits Base of Tongue into two perfectly outcome-confounded labels",
          all(site_patients.loc[lbl, "outcomes"] == [oc]
              for lbl, oc in EXPECTED["obs_site_split_pairs"].items()
              if lbl in site_patients.index),
          detail={lbl: site_patients.loc[lbl].to_dict() for lbl in EXPECTED["obs_site_split_pairs"]
                  if lbl in site_patients.index})
    check("A00.43", "obs['site'] codes Unknown as the string 'NA'", "NA" in site_patients.index,
          detail=as_str_list(site_patients.index))
    forensics["site"] = {"n_raw_labels": int(len(site_patients)),
                         "base_of_tongue_split": {lbl: site_patients.loc[lbl, "outcomes"]
                                                  for lbl in EXPECTED["obs_site_split_pairs"]
                                                  if lbl in site_patients.index},
                         "status": "FORBIDDEN - use covariate_csv 'Site of Extraction' instead"}
    TABLES["obs_site_forensics"] = site_patients.reset_index()
    note("obs['site'] must never be used raw: it splits Base of Tongue into BOT (both patients Good) "
         "and BaseofTongue (both Poor), manufacturing a confounding that does not exist, and codes "
         "Unknown as the string 'NA'. Site is taken from covariate_csv, where the 24-patient cohort "
         "spans 10 sites.")

del patient_view
forensics

## 6. Raw count validation, chunked over the sparse matrix

Row chunks of `layers['counts']` are pulled and handed to `utils.validate_counts_matrix`, which
inspects only the stored non-zeros. The matrix is never densified.

In [ ]:
counts_backed = adata.layers[COUNTS_LAYER]

row_sums = np.zeros(n_cells, dtype=np.float64)
gene_sums = np.zeros(n_genes, dtype=np.float64)
gene_cells = np.zeros(n_genes, dtype=np.int64)
nnz_total, chunk_max, chunk_min_nonzero = 0, -np.inf, np.inf
chunk_reports = []

for start in range(0, n_cells, CHUNK_ROWS):
    stop = min(start + CHUNK_ROWS, n_cells)
    chunk = counts_backed[start:stop]
    chunk = chunk.to_memory() if hasattr(chunk, "to_memory") else chunk
    if not sp.issparse(chunk):
        chunk = sp.csr_matrix(chunk)
    # raises on non-finite, negative or non-integral values without densifying
    rep = validate_counts_matrix(chunk, name=f"{COUNTS_LAYER}[{start}:{stop}]")
    chunk_reports.append(rep)
    nnz_total += int(rep["n_stored"])
    chunk_max = max(chunk_max, float(rep["max"]))
    if chunk.data.size:
        chunk_min_nonzero = min(chunk_min_nonzero, float(chunk.data.min()))
    row_sums[start:stop] = np.asarray(chunk.sum(axis=1)).ravel()
    gene_sums += np.asarray(chunk.sum(axis=0)).ravel()
    gene_cells += np.asarray((chunk > 0).sum(axis=0)).ravel()
    del chunk

zero_cells = np.where(row_sums == 0)[0]
zero_genes = np.where(gene_sums == 0)[0]
ZERO_GENES = [str(g) for g in var.index[zero_genes]]

check("A00.72", "every chunk passed utils.validate_counts_matrix "
                "(finite, non-negative, integral, never densified)",
      len(chunk_reports) == int(np.ceil(n_cells / CHUNK_ROWS))
      and all(r["densified"] is False for r in chunk_reports),
      detail={"n_chunks": len(chunk_reports)})
check("A00.73", "no all-zero cells", zero_cells.size == 0,
      detail={"n_zero_cells": int(zero_cells.size), "examples": list(obs.index[zero_cells[:10]])})
check("A00.74", "chunked nnz matches the stored nnz", nnz_total == counts_nnz,
      detail={"chunked": nnz_total, "stored": counts_nnz})
check("A00.75", "all-zero genes are reported (informational)", True,
      detail={"n_zero_genes": len(ZERO_GENES), "examples": ZERO_GENES[:20]}, blocking=False)

if ZERO_GENES:
    note(f"{len(ZERO_GENES)} genes carry zero counts across all cells and cannot contribute to any "
         "interaction; listed in audit/zero_count_genes.csv.")
    pd.DataFrame({"gene": ZERO_GENES}).to_csv(AUDIT_DIR / "zero_count_genes.csv", index=False)

COUNTS_SUMMARY = {"nnz": int(nnz_total),
                  "min_nonzero": None if chunk_min_nonzero is np.inf else float(chunk_min_nonzero),
                  "max": float(chunk_max), "n_zero_cells": int(zero_cells.size),
                  "n_zero_genes": len(ZERO_GENES),
                  "umi_per_cell_min": float(row_sums.min()),
                  "umi_per_cell_median": float(np.median(row_sums)),
                  "umi_per_cell_max": float(row_sums.max())}
TABLES["gene_detection"] = pd.DataFrame({"gene": var.index.astype(str), "counts_total": gene_sums,
                                         "n_cells_expressing": gene_cells})
TABLES["cell_umi_totals"] = pd.DataFrame({"barcode": obs.index, "counts_total": row_sums,
                                          SAMPLE_COL: obs[SAMPLE_COL].astype(str).values,
                                          CELLTYPE_COL: obs[CELLTYPE_COL].astype(str).values})
COUNTS_SUMMARY

## 7. Gene identifier semantics for `counts_data='hgnc_symbol'`

CellPhoneDB is called with `counts_data='hgnc_symbol'`, so `var_names` must be unique HGNC symbols.
This cell characterises the identifiers, rejects Ensembl-style IDs and duplicates, and quantifies how
much of the CellPhoneDB gene table this matrix can represent.

**The object carries only 10,000 genes** - a highly-variable-gene subset, not the full
transcriptome - which is recorded as a hard limitation of every interaction result downstream.

In [ ]:
var_names = pd.Index(var.index.astype(str))
HGNC_PATTERN = re.compile(r"^[A-Z0-9][A-Za-z0-9\-\.@_]*$")

dup_symbols = as_str_list(var_names[var_names.duplicated()].unique())
ensembl_like = [g for g in var_names if g.startswith("ENSG")]
non_hgnc_like = [g for g in var_names if not HGNC_PATTERN.match(g)]
symbol_like_var_cols = [c for c in var.columns
                        if c.lower() in {"gene_symbol", "symbol", "hgnc_symbol", "gene_name",
                                         "gene_ids", "feature_name"}]

check("A00.76", "var_names contain no duplicates (required for counts_data='hgnc_symbol')",
      len(dup_symbols) == 0, detail=dup_symbols[:20])
check("A00.44", "var_names are not Ensembl gene IDs", len(ensembl_like) == 0,
      detail={"n_ensembl": len(ensembl_like), "examples": ensembl_like[:5]})
check("A00.45", "var_names are HGNC-symbol shaped", len(non_hgnc_like) == 0,
      detail={"n_bad": len(non_hgnc_like), "examples": non_hgnc_like[:20]})
check("A00.46", "no separate symbol column is needed (the index IS the symbol)",
      len(symbol_like_var_cols) == 0, detail=symbol_like_var_cols, blocking=False)

with zipfile.ZipFile(CPDB_DB_PATH) as zf:
    cpdb_genes = pd.read_csv(io.BytesIO(zf.read("gene_table.csv")))
db_symbols = set(cpdb_genes["gene_name"].dropna().astype(str))
present = db_symbols & set(var_names)
coverage = len(present) / max(len(db_symbols), 1)

check("A00.47", "at least half of the CellPhoneDB gene symbols exist in var_names",
      coverage >= 0.5, detail={"db_symbols": len(db_symbols), "present": len(present),
                               "coverage": round(coverage, 4)})

GENE_ID_REPORT = {
    "n_genes": int(len(var_names)),
    "identifier_type": "HGNC symbol (var_names)",
    "is_hvg_subset": bool(len(var_names) < 20000),
    "duplicates": dup_symbols,
    "ensembl_like": len(ensembl_like),
    "non_hgnc_like": non_hgnc_like[:50],
    "mapping_required": bool(dup_symbols or ensembl_like or non_hgnc_like),
    "cpdb_gene_table_symbols": len(db_symbols),
    "cpdb_symbols_present": len(present),
    "cpdb_symbol_coverage": round(coverage, 4),
    "cpdb_symbols_absent_examples": sorted(db_symbols - set(var_names))[:25],
}
if GENE_ID_REPORT["mapping_required"]:
    raise AssertionError("STOP: var_names cannot be used as counts_data='hgnc_symbol'. "
                         f"A mapping step is required: {GENE_ID_REPORT}")

note(f"LIMITATION: the matrix carries only {len(var_names):,} genes (a highly-variable-gene subset, "
     f"not the full transcriptome). {len(present):,}/{len(db_symbols):,} ({coverage:.1%}) of "
     f"CellPhoneDB gene-table symbols are representable; the remaining {len(db_symbols) - len(present):,} "
     "ligand/receptor genes cannot be detected by ANY run in this project, so absence of an "
     "interaction is not evidence of absence.")
pd.DataFrame({"gene": sorted(db_symbols - set(var_names))}).to_csv(
    AUDIT_DIR / "cpdb_genes_absent_from_matrix.csv", index=False)
GENE_ID_REPORT

## 8. Per-sample x cell-type abundance matrix

`utils.celltype_abundance_matrix` returns **cell types as rows, samples as columns** (the
orientation `utils.core_celltypes` expects).

`per_sample_celltype_counts.csv` is written in **long** form (`batch`, `annot_6`, `n_cells`;
24 x 19 = 456 rows). That is the shape the downstream notebooks resolve by column name - `06_`
groups by patient and sums `n_cells`, `07_` pivots it and asserts 24 x 19 - so the canonical file is
long and the 24 x 19 grid is asserted as its pivot rather than stored as the primary artefact.
Notebook `01_` writes a byte-identical copy under `inputs/`, which is where `06_` looks first.

The two human-readable grids are written alongside it under distinct names, so that exactly one file
in the tree carries the canonical name (`07_` resolves it with a `**/` glob and takes the first
sorted hit).


In [ ]:
abundance_ct = celltype_abundance_matrix(obs, sample_col=SAMPLE_COL, celltype_col=CELLTYPE_COL)
abundance = abundance_ct.T                       # samples x cell types (24 x 19)

direct = (pd.crosstab(obs[SAMPLE_COL].astype(str), obs[CELLTYPE_COL].astype(str))
          .reindex(index=list(patients), columns=list(celltypes), fill_value=0).astype(int))

check("A00.48", "abundance matrix is 19 cell types x 24 samples (utils orientation)",
      abundance_ct.shape == (EXPECTED["n_celltypes"], EXPECTED["n_patients"]),
      detail=abundance_ct.shape)
check("A00.49", "abundance matrix matches a direct crosstab",
      abundance.reindex(index=list(patients), columns=list(celltypes)).equals(direct),
      detail={"n_diff": int((abundance.reindex(index=list(patients),
                                               columns=list(celltypes)) != direct).sum().sum())})
check("A00.50", "abundance matrix totals to 95,441 cells",
      int(abundance_ct.values.sum()) == EXPECTED["n_cells"], detail=int(abundance_ct.values.sum()))
check("A00.51", "abundance rows match per-patient cell counts",
      bool((abundance.sum(axis=1) == obs[SAMPLE_COL].astype(str).value_counts()
            .reindex(abundance.index)).all()), detail=None)

# Canonical long form. 01_ rebuilds this with identical code and asserts byte equality.
abundance_long = (abundance_ct.rename_axis(CELLTYPE_COL).reset_index()
                  .melt(id_vars=CELLTYPE_COL, var_name=SAMPLE_COL, value_name="n_cells")
                  .loc[:, [SAMPLE_COL, CELLTYPE_COL, "n_cells"]]
                  .sort_values([SAMPLE_COL, CELLTYPE_COL], kind="stable")
                  .reset_index(drop=True))
abundance_long["n_cells"] = abundance_long["n_cells"].astype(int)
abundance_path = assert_safe_path(AUDIT_DIR / "per_sample_celltype_counts.csv")
abundance_long.to_csv(abundance_path, index=False)

abundance.to_csv(AUDIT_DIR / "per_sample_celltype_counts_wide_24x19.csv")   # 24 x 19 grid
abundance_ct.to_csv(AUDIT_DIR / "celltype_by_sample_counts.csv")            # 19 x 24 grid

abundance_pivot = abundance_long.pivot_table(index=SAMPLE_COL, columns=CELLTYPE_COL,
                                             values="n_cells", aggfunc="sum",
                                             fill_value=0).astype(int)
check("A00.79", "per_sample_celltype_counts.csv is long form that pivots to 24 x 19",
      list(abundance_long.columns) == [SAMPLE_COL, CELLTYPE_COL, "n_cells"]
      and len(abundance_long) == EXPECTED["n_patients"] * EXPECTED["n_celltypes"]
      and abundance_pivot.shape == (EXPECTED["n_patients"], EXPECTED["n_celltypes"]),
      detail={"columns": list(abundance_long.columns), "n_rows": int(len(abundance_long)),
              "pivot_shape": abundance_pivot.shape})
check("A00.80", "the long form and the wide grid carry the same 95,441 cells",
      int(abundance_long["n_cells"].sum()) == EXPECTED["n_cells"]
      and abundance_pivot.reindex(index=list(patients), columns=list(celltypes)).equals(direct),
      detail={"long_total": int(abundance_long["n_cells"].sum())})

TABLES["celltype_by_outcome_cells"] = (
    pd.crosstab(obs[CELLTYPE_COL].astype(str), obs[OUTCOME_COL].astype(str))
      .reindex(columns=["Good", "Poor"], fill_value=0)
      .assign(total=lambda d: d.sum(axis=1), min_group=lambda d: d[["Good", "Poor"]].min(axis=1))
      .sort_values("min_group").reset_index())
abundance

### 8b. Core cell-type set: `min_cells = 10`, `min_patients = 22`

The realised set is reported and asserted against the expected six. If it differed, this notebook
stops rather than tuning the threshold.

In [ ]:
support = (abundance_ct >= PATIENT_MIN_CELLS).sum(axis=1).sort_values(ascending=False)
core = core_celltypes(abundance_ct, min_cells=PATIENT_MIN_CELLS, min_patients=CORE_MIN_PATIENTS)
direct_core = as_str_list(support.index[support >= CORE_MIN_PATIENTS])

check("A00.52", "utils.core_celltypes matches the direct threshold computation",
      as_str_list(core) == direct_core, detail={"utils": core, "direct": direct_core})
check("A00.53", "core set is exactly the 6 expected cell types (threshold NOT tuned)",
      as_str_list(core) == sorted(EXPECTED["core_celltypes"]),
      detail={"realised": core, "expected": sorted(EXPECTED["core_celltypes"])})
check("A00.54", "core cell types have the expected patient support",
      all(int(support[k]) == v for k, v in EXPECTED["core_patient_support"].items()),
      detail={k: int(support.get(k, -1)) for k in EXPECTED["core_patient_support"]})
check("A00.55", "next-ranked cell types are Mural cells (19) and CD4+ T cells (19)",
      all(int(support.get(k, -1)) == v for k, v in EXPECTED["core_next_ranked"].items()),
      detail={k: int(support.get(k, -1)) for k in EXPECTED["core_next_ranked"]})
check("A00.56", "no non-core cell type reaches the core threshold",
      set(support.index[support >= CORE_MIN_PATIENTS]) == set(core),
      detail=as_str_list(set(support.index[support >= CORE_MIN_PATIENTS]) - set(core)))

CORE_REPORT = {"min_cells": PATIENT_MIN_CELLS, "min_patients": CORE_MIN_PATIENTS,
               "core_celltypes": as_str_list(core), "n_core_celltypes": len(core),
               "n_directed_pairs": len(core) ** 2,
               "patient_support": support.astype(int).to_dict()}
TABLES["celltype_patient_support"] = (support.rename("n_patients_with_ge_min_cells")
                                      .rename_axis(CELLTYPE_COL).reset_index()
                                      .assign(is_core=lambda d: d[CELLTYPE_COL].isin(core)))
TABLES["celltype_patient_support"]

## 9. Covariate audit (`clinical_info.csv`)

The clinical table has **eight** columns and nothing else: no age, sex, stage, grade, treatment,
tumour purity or processing batch, so no confounder adjustment of any kind is possible. `Years` takes
only the values 2, 3 and 5 and `Clinical outcome` is a deterministic function of it (2 -> Poor,
3 and 5 -> Good), so the file carries no censoring information and no survival analysis is possible.

In [ ]:
clinical = pd.read_csv(CLINICAL_CSV, encoding="utf-8-sig")
clinical.columns = [c.strip() for c in clinical.columns]

check("A00.57", "clinical_info.csv has exactly the 8 known columns",
      list(clinical.columns) == EXPECTED["clinical_columns"],
      detail={"observed": list(clinical.columns), "expected": EXPECTED["clinical_columns"]})

COVARIATE_KEYWORDS = ["age", "sex", "gender", "stage", "grade", "tnm", "treatment", "therapy",
                      "chemo", "radiation", "purity", "batch", "chemistry", "platform", "date",
                      "margin", "pni", "lvi", "smoking", "recurrence", "metastasis", "censor",
                      "followup", "follow_up"]
found_cov = [c for c in clinical.columns
             if any(k in c.lower().replace(" ", "_") for k in COVARIATE_KEYWORDS)]
check("A00.58", "no biological or technical covariate columns exist in clinical_info.csv",
      len(found_cov) == 0, detail=found_cov)
check("A00.59", "config records site_of_extraction as the only available covariate",
      as_str_list(CFG.get("covariates_available", [])) == ["site_of_extraction"],
      detail=CFG.get("covariates_available"))
check("A00.60", "config records survival-aware analysis as not possible",
      CFG.get("survival_aware_analysis") == "not_possible",
      detail=CFG.get("survival_aware_analysis"))
note("LIMITATION: clinical_info.csv contains no age, sex, stage/grade, treatment, tumour purity or "
     "processing-batch information. No covariate adjustment, matching or stratification beyond "
     "anatomical site is possible; every outcome contrast in this project is unadjusted.")

clinical["Sample ID"] = clinical["Sample ID"].astype(str).str.strip()
sc_patients, cl_patients = set(patients), set(clinical["Sample ID"])
only_clinical = as_str_list(cl_patients - sc_patients)
only_sc = as_str_list(sc_patients - cl_patients)

check("A00.61", "every single-cell patient is present in clinical_info.csv", not only_sc, detail=only_sc)
check("A00.62", "P14 is the only clinical-only patient and has no single-cell data",
      only_clinical == [EXPECTED["clinical_only_patient"]]
      and int(clinical.loc[clinical["Sample ID"] == EXPECTED["clinical_only_patient"],
                           "Single Cell RNA Datasets"].iloc[0]) == 0,
      detail={"clinical_only": only_clinical})
check("A00.63", "the analysis cohort is n = 24 patients",
      len(sc_patients & cl_patients) == EXPECTED["n_patients"], detail=len(sc_patients & cl_patients))

clin_sc = clinical[clinical["Sample ID"].isin(sc_patients)].copy()
clin_sc["derived_outcome"] = clin_sc["Sample ID"].map(patient_outcomes)
check("A00.64", "clinical_info 'Clinical outcome' agrees with the obs-derived outcome for all 24",
      bool((clin_sc["Clinical outcome"].astype(str).str.strip() == clin_sc["derived_outcome"]).all()),
      detail=clin_sc.loc[clin_sc["Clinical outcome"].astype(str).str.strip() != clin_sc["derived_outcome"],
                         ["Sample ID", "Clinical outcome", "derived_outcome"]].to_dict("records"))

In [ ]:
# ---- Site of extraction x outcome, and the Years -> outcome determinism --------------
check("A00.65", "the site covariate is taken from covariate_csv, not from the forbidden obs['site']",
      SITE_COL in clin_sc.columns, detail={"covariate_csv": str(CLINICAL_CSV), "column": SITE_COL})

site_tab = (pd.crosstab(clin_sc[SITE_COL].astype(str).str.strip(), clin_sc["derived_outcome"])
            .reindex(columns=["Good", "Poor"], fill_value=0))
site_tab["total"] = site_tab.sum(axis=1)
site_tab["single_outcome"] = (site_tab["Good"] == 0) | (site_tab["Poor"] == 0)

check("A00.66", "the 24-patient cohort spans 10 anatomical sites in covariate_csv "
                "(obs['site'] would give 11 by splitting Base of Tongue)",
      len(site_tab) == EXPECTED["csv_site_labels"],
      detail={"csv_sites": len(site_tab), "obs_site_labels": EXPECTED["obs_site_labels"],
              "sites": as_str_list(site_tab.index)})

for site, (g, p) in EXPECTED["site_by_outcome"].items():
    check(f"A00.67.{site.replace(' ', '_')}", f"site '{site}' has {g} Good / {p} Poor patients",
          site in site_tab.index and int(site_tab.loc[site, "Good"]) == g
          and int(site_tab.loc[site, "Poor"]) == p,
          detail=site_tab.loc[site].to_dict() if site in site_tab.index else "site absent")

fully_confounded = as_str_list(site_tab.index[site_tab["single_outcome"]])
check("A00.77", "anatomical site is confounded with outcome (recorded, not corrected)",
      len(fully_confounded) > 0, detail={"single_outcome_sites": fully_confounded}, blocking=False)
note("CONFOUNDING: anatomical site is severely imbalanced against outcome - all 4 Maxilla patients "
     "are Poor, while Parotid Gland (3), Larynx (2) and Tongue (2) are exclusively Good; only Base "
     "of Tongue (2 Good / 2 Poor) is balanced. Site-specific biology cannot be separated from "
     "outcome at n = 24, and site cannot be adjusted for. Reported as a limitation.")

years = pd.to_numeric(clin_sc["Years"], errors="raise").astype(int)
years_levels = {int(y) for y in years.unique()}
years_map = clin_sc.assign(Years=years).groupby("Years")["derived_outcome"].agg(lambda s: sorted(set(s)))

check("A00.78", "Years takes only the values {2, 3, 5}", years_levels == EXPECTED["years_levels"],
      detail=sorted(years_levels))
check("A00.68", "clinical_outcome is a deterministic function of Years (2->Poor, 3&5->Good)",
      all(v == [EXPECTED["years_to_outcome"][int(k)]] for k, v in years_map.items()),
      detail={int(k): v for k, v in years_map.items()})
note("LIMITATION: 'Years' is a 3-level coarse survival bin (2/3/5) that determines 'Clinical "
     "outcome' exactly (2 -> Poor, 3 & 5 -> Good). There is no event indicator, no follow-up time "
     "and no censoring flag, so no survival-aware (Kaplan-Meier / Cox) analysis is possible; the "
     "outcome is a binary label only.")

# ---- obs['years_of_survival'] must agree with the CSV on all 24 patients ----------
obs_years = (obs.groupby(SAMPLE_COL, observed=True)["years_of_survival"]
             .agg(lambda s: sorted({int(float(x)) for x in s})))
check("A00.81", "obs['years_of_survival'] is single-valued per patient",
      bool((obs_years.map(len) == 1).all()),
      detail={k: v for k, v in obs_years.items() if len(v) != 1})
obs_years = obs_years.map(lambda v: v[0])
csv_years = clin_sc.set_index("Sample ID")["Years"].astype(int)
years_disagree = {p: {"obs": int(obs_years[p]), "csv": int(csv_years[p])}
                  for p in csv_years.index if int(obs_years.get(p, -1)) != int(csv_years[p])}
check("A00.82", "obs['years_of_survival'] equals the clinical_info 'Years' on all 24 patients",
      len(years_disagree) == 0 and len(csv_years) == EXPECTED["n_patients"],
      detail={"n_patients": int(len(csv_years)), "disagreements": years_disagree})
TABLES["years_cross_check"] = pd.DataFrame({SAMPLE_COL: csv_years.index,
                                            "obs_years_of_survival": obs_years.reindex(csv_years.index).values,
                                            "clinical_csv_years": csv_years.values,
                                            "derived_outcome": clin_sc.set_index("Sample ID")
                                                               ["derived_outcome"].reindex(csv_years.index).values})

TABLES["site_by_outcome"] = site_tab.reset_index()
TABLES["clinical_cohort"] = clin_sc
TABLES["years_by_outcome"] = (pd.crosstab(years, clin_sc["derived_outcome"])
                              .reindex(columns=["Good", "Poor"], fill_value=0).reset_index())
site_tab

### 9b. `result_precision` does not reach the interaction scores

`config.yaml` sets `cpdb_result_precision: 6`. Verified against the **installed** package, not the
documentation: `cellphonedb/utils/scoring_utils.py` hardcodes `np.around(lr_outer, 3)` and
`result_precision` never appears in that file, so it is never passed into the scoring path. It
reaches only `means`, `significant_means`, `deconvoluted` and `deconvoluted_percents`.

**Consequence, stated plainly:** interaction scores - the primary endpoint of this project - stay
quantised at `1e-3` no matter what `result_precision` is set to. Raising it does not remove ties from
the primary endpoint, and the tie handling in the downstream tests must not assume otherwise. The
setting is kept at 6 because it genuinely improves the other four tables.


In [ ]:
import cellphonedb

scoring_path = Path(cellphonedb.__file__).parent / "utils" / "scoring_utils.py"
scoring_src = scoring_path.read_text()
hardcoded_rounding = [f"{i}: {l.strip()}" for i, l in enumerate(scoring_src.splitlines(), 1)
                      if "np.around(lr_outer" in l]
precision_occurrences = scoring_src.count("result_precision")

# Non-blocking: if a future cellphonedb release wires result_precision into the scoring path this
# should be noticed and recorded, not treated as a reason to fail the audit.
check("A00.83", "scoring_utils.py rounds interaction scores with a hardcoded 3 decimals",
      len(hardcoded_rounding) == 1 and hardcoded_rounding[0].endswith("np.around(lr_outer, 3)"),
      detail={"file": str(scoring_path), "matches": hardcoded_rounding}, blocking=False)
check("A00.84", "result_precision never appears in the interaction-score path",
      precision_occurrences == 0,
      detail={"file": str(scoring_path), "occurrences": precision_occurrences}, blocking=False)

PRECISION_FINDING = {
    "config_cpdb_result_precision": CFG.get("cpdb_result_precision"),
    "verified_against": "installed package",
    "cellphonedb_version": pkg_versions.get("cellphonedb"),
    "evidence_file": str(scoring_path),
    "hardcoded_rounding": hardcoded_rounding,
    "result_precision_occurrences_in_file": int(precision_occurrences),
    "applies_to": ["means", "significant_means", "deconvoluted", "deconvoluted_percents"],
    "does_not_apply_to": ["interaction_scores"],
    "consequence": ("interaction scores stay quantised at 1e-3 regardless of result_precision; "
                    "raising the setting does NOT remove ties from the primary endpoint"),
}
note("result_precision does NOT apply to interaction_scores: "
     f"{scoring_path} hardcodes np.around(lr_outer, 3) and never receives result_precision. "
     "The primary endpoint is therefore quantised at 1e-3 whatever the configuration says, so "
     "ties in interaction scores are expected and must be handled by the downstream tests rather "
     "than assumed away. result_precision: 6 is kept because it does improve means, "
     "significant_means, deconvoluted and deconvoluted_percents.")
PRECISION_FINDING


## 10. Write audit artefacts and the initial manifest

In [ ]:
for name, df in TABLES.items():
    out = assert_safe_path(AUDIT_DIR / f"{name}.csv")
    pd.DataFrame(df).to_csv(out, index=False)

AUDIT_REPORT = {
    "notebook": "00_audit_inputs_and_sources",
    "started_utc": RUN_STARTED,
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "config_path": str(CONFIG_PATH),
    "config_hash": CFG_HASH,
    "config": CFG,
    "environment": ENVIRONMENT,
    "prerequisites_missing": PREREQ_MISSING,
    "inputs": inventory.reset_index().to_dict("records"),
    "cpdb_database_members": CPDB_DB_MEMBERS,
    "matrix": MATRIX_INFO,
    "counts_summary": COUNTS_SUMMARY,
    "gene_identifiers": GENE_ID_REPORT,
    "cohort": {"n_cells": int(n_cells), "n_genes": int(n_genes),
               "n_patients": int(len(patients)), "n_celltypes": int(len(celltypes)),
               "celltypes": list(celltypes),
               "patient_outcomes": patient_outcomes.to_dict(),
               "good_patients": GOOD_PATIENTS, "poor_patients": POOR_PATIENTS,
               "n_good_cells": n_good_cells, "n_poor_cells": n_poor_cells},
    "forbidden_obs_columns_forensics": forensics,
    "result_precision_finding": PRECISION_FINDING,
    "core_celltypes": CORE_REPORT,
    "covariates": {
        "clinical_columns": list(clinical.columns),
        "covariates_available": ["site_of_extraction (confounded, unusable for adjustment)"],
        "covariates_absent": ["age", "sex", "stage", "grade", "treatment", "tumour purity",
                              "processing batch", "censoring / follow-up time"],
        "site_by_outcome": site_tab.drop(columns=["single_outcome"]).to_dict("index"),
        "sites_confounded_with_outcome": fully_confounded,
        "years_levels": sorted(years_levels),
        "years_to_outcome": {int(k): v[0] for k, v in years_map.items()},
        "clinical_only_patients": only_clinical,
        "survival_analysis_possible": False,
    },
    "limitations": NOTES,
    "checks": CHECKS,
    "artifacts": sorted(str(p.relative_to(OUTPUT_ROOT)) for p in AUDIT_DIR.rglob("*") if p.is_file()),
}
report_path = assert_safe_path(AUDIT_DIR / "audit_report.json")
report_path.write_text(json.dumps(AUDIT_REPORT, indent=2, default=str))

manifest_targets = [Path(v) for v in INPUT_FILES.values()] + \
                   sorted(p for p in AUDIT_DIR.rglob("*") if p.is_file())
write_manifest(manifest_targets, MANIFEST_PATH)

check("A00.69", "MANIFEST.sha256 written", MANIFEST_PATH.is_file(), detail=str(MANIFEST_PATH))
check("A00.70", "audit_report.json written", report_path.is_file(), detail=str(report_path))
check("A00.71", "per_sample_celltype_counts.csv written (long form pivoting to 24 x 19)",
      abundance_path.is_file()
      and abundance_pivot.shape == (EXPECTED["n_patients"], EXPECTED["n_celltypes"]),
      detail={"path": str(abundance_path), "n_rows": int(len(abundance_long)),
              "pivot_shape": abundance_pivot.shape})
sorted(p.name for p in AUDIT_DIR.iterdir())

## Gate 0

The notebook stops here unless every blocking assertion passed. `gate0_success.json` is the
machine-readable record that notebook 01 requires before it builds anything.

In [ ]:
failed_blocking = [c for c in CHECKS if not c["passed"] and c["blocking"]]
failed_soft = [c for c in CHECKS if not c["passed"] and not c["blocking"]]

# ---- Gate 0 explicit final assertions ------------------------------------------------
assert n_cells == 95441, "cell count"
assert len(patients) == 24 and len(celltypes) == 19, "cohort shape"
assert len(GOOD_PATIENTS) == 13 and len(POOR_PATIENTS) == 11, "patient split"
assert (n_good_cells, n_poor_cells) == (46260, 49181), "cell split"
assert set(patient_outcomes.unique()) == {"Good", "Poor"}, "outcome levels"
assert obs.index.is_unique and var.index.is_unique, "identifier uniqueness"
assert not GENE_ID_REPORT["mapping_required"], "hgnc_symbol usability"
assert COUNTS_SUMMARY["n_zero_cells"] == 0, "all-zero cells"
assert as_str_list(core) == sorted(EXPECTED["core_celltypes"]), "core cell-type set"
assert list(clinical.columns) == EXPECTED["clinical_columns"], "clinical schema"
assert not years_disagree, "obs years_of_survival vs clinical_info Years"
assert abundance_pivot.shape == (24, 19), "per_sample_celltype_counts.csv shape"
assert not failed_blocking, f"blocking checks failed: {[c['id'] for c in failed_blocking]}"

gate0_passed = True

GATE0 = {
    "gate": "gate0_audit",
    "notebook": "00_audit_inputs_and_sources",
    "passed": gate0_passed,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "config_hash": CFG_HASH,
    "input_sha256": {k: inventory.loc[k, "sha256"] for k in inventory.index},
    "n_checks": len(CHECKS),
    "n_failed_blocking": len(failed_blocking),
    "n_failed_soft": len(failed_soft),
    "soft_failures": [c["id"] for c in failed_soft],
    "prerequisites_missing": PREREQ_MISSING,
    "cohort": {"n_cells": int(n_cells), "n_genes": int(n_genes),
               "good_patients": GOOD_PATIENTS, "poor_patients": POOR_PATIENTS,
               "n_good_cells": n_good_cells, "n_poor_cells": n_poor_cells,
               "celltypes": list(celltypes)},
    "core_celltypes": as_str_list(core),
    "result_precision_finding": PRECISION_FINDING,
    "per_sample_celltype_counts": str(abundance_path.relative_to(OUTPUT_ROOT)),
    "audit_report": str(report_path.relative_to(OUTPUT_ROOT)),
    "manifest": str(MANIFEST_PATH.relative_to(OUTPUT_ROOT)),
}
gate0_path = assert_safe_path(AUDIT_DIR / "gate0_success.json")
gate0_path.write_text(json.dumps(GATE0, indent=2, default=str))
log_event({"event": "gate0", **{k: v for k, v in GATE0.items() if k != "input_sha256"}})

adata.file.close()

print(f"GATE 0 PASSED - {len(CHECKS)} checks, {len(failed_soft)} non-blocking failures "
      f"{[c['id'] for c in failed_soft]}")
if PREREQ_MISSING:
    print(f"ACTION REQUIRED before the deliverable notebooks: install {', '.join(PREREQ_MISSING)} "
          "into the cpdb environment.")
GATE0